### 1-1 Import sales CSV file into a dataframe: 

In [ ]:
import pandas as pd
df = pd.read_csv('retail_sales_dirty_dataset.csv')
display(df.head(10))

### 1-2 Explore input data structure, null counts, number of rows, columns list and data types:

In [ ]:
print(df.info())

### 1-3 Missing Values Count:

In [ ]:
def missing_values_count(data_frame):
  """
    Returns the number of missing values by columns of a dataset
    that contain at least one missing value. 
  """
  missing_count = data_frame.isnull().sum()
  missing_count = missing_count[missing_count > 0 ]
  return missing_count

print("The number of missing values in each column are as follows:")
print(missing_values_count(df) )

### 1-4 Identifying Invalid Values :
### 1-4-1 Identifying invalid string values for "date" column (can not be converted to date) :

In [ ]:
## invalid yyyy-mm-dd values. Can not convert string to date
df_invalid_dates = df[pd.to_datetime(df['date'],format ='mixed', errors='coerce').isna()]
print("Invalid dates entry (to be removed in data Cleaning stage)")
print(df_invalid_dates[['order_id','date','price']])

### 1-4-2 Identifying non-numeric sales values (can be converted to numeric):

In [ ]:
 ## only column "sales" data type is string instead of numeric. Check if those non-numeric strings can be converted to numeric.

df1=  df[pd.to_numeric(df['sales'] , errors='coerce').isna()]
df1 = df1[df1['sales'].notnull()]
print("sales values that need to be modified before converting to numeric:")
display(df1[['order_id','date','price','sales']])


## 2 - Data Cleaning
#### In this section we will clean data, converting data types, removing irrevelant columns, handling missing info, handling wrong values, removing rows with invalid data, etc.

### 2-1 Removing Irrevelant Columns:

In [ ]:
## there is a column irrelevant_column which its values have no business value. We will not use this column in any analysis.
df.drop(columns=['irrelevant_column'], inplace = True)
display(df.head(10))

### 2-2 Converting data types:
 - Drop rows with invalid date coluumn, then convert date into datetime
 - Convert sales values from string to numeric




In [ ]:
## The df_clen dataset will be our cleaned, modified, corrected dataset to be loaded into target database.

## 1- drop rows with invalid date values to be converted to datetime.
df_clean = df.drop( index = df.loc[pd.to_datetime(df['date'],format ='mixed', errors='coerce').isna()].index)

## now converting the column "date" datatype from string to datetime.
df_clean['date'] = pd.to_datetime(df_clean['date'],format ='mixed')

## now converting the column "sales" datatype from string to float64.
df_clean['sales'] = pd.to_numeric(df_clean['sales'].str.replace(r'[$,]','',regex=True),errors='coerce')

print("All Data tpes are corrected into dataframe clen-df")
display(df_clean.info())

print("The number of missing values in df_clean data frame are as follows:")
print(missing_values_count(df_clean) )

## 2-3 Handling Unexpected Missing, Negative, and Wrong Values

- In this step, we will prepare data for analytics and target environment.
- Important functions 

In [ ]:
#  This function identifies unexpected negative numbers within a given colum of a data frame 

def unexpected_negative_values(data_frame , column_name):
 df_invalid_negative_val =  data_frame.loc[data_frame[column_name] <0 ,['order_id','date','quantity','price','sales','cost']] 
 return df_invalid_negative_val 




### 2-3-1 'region' Column:  Validation, Cleaning, and Missing Value Handling 
- text format correction
- N/A default values for nulls
- Stripping trailer white spaces.

In [ ]:
# Handling 'region' missing values and changing values to proper format.

df_clean['region'] = df_clean['region'].fillna('N/A')
print("Region column values before cleaning:")
print(df_clean.groupby(['region']).size())

df_clean['region'] = df_clean['region'].str.title()
df_clean['region'] = df_clean['region'].str.strip()
print("\nRegion column values after cleaning:")
print(df_clean.groupby(['region']).size())


## For missing sales values, if quantity and price values are valie '>0': 
#  populate sales value as  sales = quantity x price  
#df_clean['sales'] = pd.to_numeric(df_clean['sales'].str.replace(r'[$,]','',regex=True),errors='coerce')
#print( df_clean.loc[df_clean['date'].isna()] )
#print( df_clean.loc[df_clean['sales'].isna(), ['order_id','date','quantity','price','sales','cost']])

###  2-3-2 'price' Column: Validation, Cleaning, and Missing Value Handling 

In [ ]:
import logging

logging.basicConfig(
  filename= 'Logfile-Retail_project_Sales.log',
  level = logging.INFO,
  format = '%(asctime)s - %(levelname)s - %(message)s', force=True
) 

''' This function is printing number of missing rows, number of negative values,
    and number of rows with wrong data by validating sales = price x quantity 
    for a given data frame having the columns price, quantity, and sales
'''

def report_quality(dtframe,column_name):
  print(f"Number of rows with missing {column_name} values:",dtframe[column_name].isnull().sum()) 
  print(f"Number of rows with negative {column_name} values:",(dtframe[column_name]<0).sum() ) 
  print(
    "Number of rows unmatching (price, quantity, sales) values:", 
    (dtframe['sales'].round(2) !=
    (dtframe['price'] * dtframe['quantity']).round(2)).sum()
  )

report_quality(df_clean,'price')

## convert negaive to positive for negative values:
df_clean['price'] = df_clean['price'].abs()

## Populate missing price values by subtracting sales by quantity if both values are populated and positive:
mask = (
   df_clean['price'].isna() & 
   df_clean['sales'].notna() &
   df_clean['quantity'].notna() &
   (df_clean['quantity'] != 0) 
)

df_clean.loc[mask,'price']  = (df_clean.loc[mask,'sales'] / df_clean.loc[mask,'quantity'] ).round(2)


## if price is still null, it means quantity or sales are null. This is a bad record and should be removed.
#display(df_clean[df_clean['price'].isna()])
for index,row in  df_clean[df_clean['price'].isna()].iterrows():
  logging.warning(
  f"Record (index = {index} Removed,"
  f"order-id:{row['order_id']},"
  f"Reason: price and quantity or sales values are null.can not calculate price."
  )

df_clean= df_clean.dropna(subset=['price'])

print("\n Records with price values that could not be calculated were removed.:\n",missing_values_count(df_clean[['price']]) )                                    

report_quality(df_clean,'price')  

### 2-3-3 'quantity' column:  Validation, Cleaning, and Missing Value Handling 

In [ ]:
report_quality(df_clean,'quantity')  
## convert negaive to positive for negative values:
#df_clean['quantity'] = df_clean['quantity'].abs()

print("min:",df_clean['quantity'].min())
print("max:",df_clean['quantity'].max())

if df_clean['quantity'].min()>0 and df_clean['quantity'].max()<999999:
  print("no issue in the quantity colum.\n")  

report_quality(df_clean,'quantity')  


### 2-3-4 'sales' column: Validation, Cleaning and Missing Value Handling

In [ ]:
report_quality(df_clean,'sales')

# Handling Negative values:

df_clean['sales'] = df_clean['sales'].abs()

print("\nAfter fixing negative values:")
report_quality(df_clean,'sales')


# Populate missing sales values by quantity x price
mask = (
   df_clean['price'].notna() & 
   df_clean['quantity'].notna() &
   df_clean['sales'].isna() &
  (df_clean['quantity'] != 0) 
)
df_clean.loc[mask,'sales']  = (df_clean.loc[mask,'quantity'] * df_clean.loc[mask,'price'] ).round(2)

print("\nAfter populating missing sales values:")
report_quality(df_clean,'sales')

# Modify wrong sales values by pllying sales = quantity x price:

mask = (
   df_clean['price'].notna() & 
   df_clean['quantity'].notna() &
   (df_clean['sales'].round(2) !=  (df_clean['price'] * df_clean['quantity']).round(2)) &
   (df_clean['quantity'] != 0) 
)
df_clean.loc[mask,'sales']  = (df_clean.loc[mask,'quantity'] * df_clean.loc[mask,'price'] ).round(2)



#print("\nAfter cleaning: Number of rows with missing sales values:",df_clean['sales'].isnull().sum()) 
print("\nAfter cleaning:")
report_quality(df_clean,'sales')

## 3 - Export Clean Data file
 - Clean data from dataframe df_clean is being exported as retail_sales_clean_dataset.csv
 - This dataset is ready to be used for Analytics & Reporting.


In [ ]:
output_file = "retail_sales_clean_dataset.csv"

df_clean.to_csv(output_file, index=False)

print(f"Clean dataset successfully exported to: {output_file}")